# 🔬 Lab 04: Production Similarity Search & Benchmarking (HNSW vs IVF-PQ vs Flat)

---

## 🎯 Executive Overview & Production Challenge
At enterprise scale (1M+ vectors), exact brute-force search is computationally intractable ($O(N \cdot d)$ per query). Engineers must balance the **Pareto Trade-off** between:
1. **Recall@K:** The percentage of true top-K neighbors correctly retrieved.
2. **Search Latency (p95/p99):** Query execution speed in milliseconds.
3. **Memory Footprint (RAM):** Memory consumed by dense Float32 vectors vs compressed quantized vectors.

### Comparison of Index Architectures:
| Index Type | Search Time Complexity | Memory Consumption | Recall@K Accuracy | Best Production Fit |
| :--- | :--- | :--- | :--- | :--- |
| **Flat (Exact)** | $O(N \cdot d)$ | High (100% Float32) | 100% (Ground Truth) | Small catalogs (<50K vectors) |
| **IVF-Flat** | $O(\frac{N}{K_{clusters}} \cdot d)$ | Medium | 90 - 95% | Medium scale batch jobs |
| **HNSW (Graph)** | $O(\log N)$ | Higher (Vectors + Graph Edges) | 98 - 99.5% | Low-latency real-time RAG |
| **Scalar Quantized (SQ-int8)** | $O(\log N)$ | 75% RAM Reduction (int8) | 97 - 99% | Enterprise scale-out memory optimization |


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic seed
np.random.seed(42)

# Generate a synthetic scale dataset of 10,000 vectors in 384 dimensions
NUM_VECTORS = 10000
DIM = 384
NUM_QUERIES = 50

print(f"[*] Generating {NUM_VECTORS} syntheticFloat32 vectors ({DIM} dimensions)...")
corpus_vectors = np.random.randn(NUM_VECTORS, DIM).astype(np.float32)
# Normalize to unit sphere (so Dot Product == Cosine Similarity)
corpus_vectors /= np.linalg.norm(corpus_vectors, axis=1, keepdims=True)

query_vectors = np.random.randn(NUM_QUERIES, DIM).astype(np.float32)
query_vectors /= np.linalg.norm(query_vectors, axis=1, keepdims=True)

print(f"[✓] Corpus RAM size: {corpus_vectors.nbytes / (1024 * 1024):.2f} MB")


---
## Step 1: Benchmarking Exact Flat Search (Ground Truth)
Exact search calculates dot product against all $N$ vectors. It provides 100% recall but scales linearly.


In [ ]:
def exact_flat_search(queries, corpus, top_k=10):
    t0 = time.perf_counter()
    # Matrix multiplication: queries (Q x D) @ corpus.T (D x N) -> (Q x N)
    sim_matrix = np.dot(queries, corpus.T)
    top_indices = np.argsort(sim_matrix, axis=1)[:, ::-1][:, :top_k]
    t1 = time.perf_counter()
    avg_latency = (t1 - t0) / len(queries) * 1000 # ms
    return top_indices, avg_latency

ground_truth_indices, flat_latency = exact_flat_search(query_vectors, corpus_vectors, top_k=10)
print(f"[✓] Exact Flat Search Latency: {flat_latency:.3f} ms per query (Recall = 100.0%)")


---
## Step 2: Benchmarking HNSW Graph Retrieval
We simulate HNSW graph retrieval behavior using Qdrant with varying `ef_search` parameters to observe the Latency vs Recall trade-off.


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, SearchParams

client = QdrantClient(":memory:")
client.recreate_collection(
    collection_name="benchmark_hnsw",
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE)
)

# Upload synthetic corpus
points = [
    PointStruct(id=i, vector=corpus_vectors[i].tolist())
    for i in range(NUM_VECTORS)
]
client.upsert(collection_name="benchmark_hnsw", points=points)
print("[✓] Loaded 10,000 vectors into Qdrant HNSW graph.")


In [ ]:
ef_values = [8, 16, 32, 64, 128]
hnsw_latencies = []
hnsw_recalls = []

for ef in ef_values:
    retrieved_indices = []
    t0 = time.perf_counter()
    for q in query_vectors:
        if hasattr(client, "query_points"):
            res = client.query_points(
                collection_name="benchmark_hnsw",
                query=q.tolist(),
                limit=10,
                search_params=SearchParams(hnsw_ef=ef)
            )
            hits = res.points
        else:
            hits = client.search(
                collection_name="benchmark_hnsw",
                query_vector=q.tolist(),
                limit=10,
                search_params=SearchParams(hnsw_ef=ef)
            )
        retrieved_indices.append([h.id for h in hits])
    t1 = time.perf_counter()
    
    latency = (t1 - t0) / len(query_vectors) * 1000
    
    # Calculate Recall@10 against Ground Truth
    recalls = []
    for q_idx in range(len(query_vectors)):
        gt = set(ground_truth_indices[q_idx])
        ret = set(retrieved_indices[q_idx])
        recalls.append(len(gt.intersection(ret)) / 10.0)
    
    avg_recall = np.mean(recalls) * 100.0
    hnsw_latencies.append(latency)
    hnsw_recalls.append(avg_recall)
    print(f"ef_search: {ef:3d} | Latency: {latency:6.3f} ms | Recall@10: {avg_recall:5.1f}%")


---
## Step 3: Visualizing the Latency-Recall Pareto Frontier
We plot the empirical Pareto curve showing how increasing `ef_search` improves recall at the expense of latency.


In [ ]:
plt.figure(figsize=(8, 4.5), dpi=120)
plt.plot(hnsw_latencies, hnsw_recalls, marker='s', lw=2, color='#10b981', label='HNSW Graph Index')
plt.scatter([flat_latency], [100.0], color='#ef4444', s=90, label='Exact Flat Search (100% Recall)', zorder=5)

for i, ef in enumerate(ef_values):
    plt.annotate(f"ef={ef}", (hnsw_latencies[i], hnsw_recalls[i]), 
                 textcoords="offset points", xytext=(8, -5), fontsize=8)

plt.title('Production Pareto Frontier: Query Latency vs. Recall@10', fontsize=11, fontweight='bold')
plt.xlabel('Average Latency per Query (ms)', fontsize=9)
plt.ylabel('Recall@10 Accuracy (%)', fontsize=9)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


---
## Step 4: Scalar Quantization (float32 -> int8)
Quantization maps 32-bit floats (-1.0 to 1.0) into 8-bit integers (-128 to 127), reducing RAM consumption by **75%**.


In [ ]:
# Compute quantization scale
def quantize_to_int8(vectors):
    v_min = vectors.min()
    v_max = vectors.max()
    scale = 255.0 / (v_max - v_min)
    int8_vecs = np.round((vectors - v_min) * scale - 128).astype(np.int8)
    return int8_vecs, scale, v_min

int8_corpus, scale, v_min = quantize_to_int8(corpus_vectors)
print(f"Float32 Original Size: {corpus_vectors.nbytes / 1024:.1f} KB")
print(f"Int8 Quantized Size:   {int8_corpus.nbytes / 1024:.1f} KB")
print(f"[✓] Compression Ratio: {corpus_vectors.nbytes / int8_corpus.nbytes:.1f}x Memory Reduction!")


---
## 🏁 Production Engineering Takeaways & Exercises
1. **Tune `ef_search` dynamically:** In production, use `ef_search=64` for critical diagnostic queries (99% recall), and `ef_search=16` for high-throughput background tasks.
2. **Enable Scalar Quantization:** In databases with over 5 million vectors, scalar quantization reduces hardware hosting costs by 75% with less than 1.5% drop in Recall@10.
